# Step 2b: Model 2 ("Long-term") sequence inputs

Turns the cleaned step-1 splits into the arrays the long-term model consumes:

- **History tensors** `[n_slates, MAX_LEN]`: the most recent MAX_LEN actions per user,
  left-padded, oldest -> newest (the last column is the most recent action).
  Per position: item index, brand / L4-category / shop index, action type,
  searched-or-not flag, the query that led to the action, days before the search,
  and a log-scale time bucket.
- **Candidate arrays** (one entry per candidate, not padded): which slate row it
  belongs to, its position in the step-1 list, item / attribute / name indices, label,
  and has_metadata. `(slate_id, pos)` is the shared join key with the scalar features.
- **Term tables** for text: past queries (`qry_terms`) and candidate names (`name_terms`)
  are stored once per split as padded rows of term indices; the history and candidate
  arrays point into them. In the model, a query or name vector = mean of its term
  embeddings over non-zero entries (an all-zero row means "no query" / "no name").

Index convention for every embedding vocabulary: 0 = padding, 1 = unknown
(not in the train vocabulary, or product has no metadata), 2.. = known values.

Vocabularies (items, attributes, terms) are built from the TRAIN split only, so val/test
honestly measure how the model handles items and words it has never seen.

Not stored, computed inside the model from saved arrays:
  weighted_action_score = ACTION_WEIGHT[hist_action]   (weights saved in vocab.json)
  decayed_recency       = decayed_recency(hist_days)   (half-life tuned on validation)

## Setup (Google Colab)

1. In Google Drive, add a shortcut to the shared data folder in **My Drive** (right-click the folder → Organize → Add shortcut).
2. Run the cell below to mount Drive, then set `DATA_DIR` in the Settings cell to that folder's path.

Inputs read from `DATA_DIR`: `train_sampled.parquet`, `val.parquet`, `test.parquet`, `product_metadata_clean.parquet` (step 1 outputs).
Outputs written to `DATA_DIR/model2_inputs/`. Standard Colab RAM (about 12 GB) is enough; no GPU needed.

In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DATA_DIR to a local folder in the Settings cell.")

Mounted at /content/drive


In [2]:
import json
import os
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

## Settings
Each value below is a modelling decision that should be justified in the report.

In [3]:
# >>> Change this to the shared folder's path in your Drive <<<
DATA_DIR = Path("/content/drive/MyDrive/JDSearch")
OUT_DIR = DATA_DIR / "model2_inputs"

MAX_LEN = 188          # 75th percentile of history length (interim report Part 2 §2.4.1)
MIN_ITEM_COUNT = 2     # items seen once in train get the shared "unknown" embedding
ATTR_COLS = ["brand_id", "cate_id_4", "shop_id"]

TERM_MIN_COUNT = 2     # query/name terms seen once in train map to "unknown"
QRY_MAX_TERMS = 8      # 99th percentile of query length (interim report Part 2 §2.6)
NAME_MAX_TERMS = 50    # 99th percentile of product name length (p95 = 42, p99 = 50)

ACTIONS = ["CLICK", "FLW", "CART", "ORD"]                       # action ids 1..4 (0 = padding)
ACTION_WEIGHT = np.array([0, 1, 2, 5, 10], dtype=np.float32)    # indexed by action id
TIME_BINS_DAYS = [1 / 24, 1, 3, 7, 14, 30, 60, 90, 180]         # time buckets 1..10 (0 = padding)

PAD, UNK = 0, 1
NO_QUERY = "-1"
TERM_SEP = "\x18"

HIST_COLS = ["history_wid_list", "history_type_list", "history_has_query",
             "history_days_before", "history_qry_list"]
CAND_COLS = ["candidate_wid_list", "candidate_label_list", "candidate_has_metadata"]

## Helpers

In [4]:
def read_split(path):
    return pq.read_table(path, columns=["slate_id"] + HIST_COLS + CAND_COLS)


def flat(table, col):
    """List column -> (flat values as a pyarrow array, list lengths as numpy)."""
    arr = table.column(col).combine_chunks()
    return arr.flatten(), arr.value_lengths().to_numpy(zero_copy_only=False)


def list_lengths(arr):
    return pc.fill_null(pc.list_value_length(arr), 0).to_numpy(zero_copy_only=False)


class Lookup:
    """Maps raw IDs to embedding indices with a sorted binary search; misses -> UNK."""

    def __init__(self, keys, values=None):
        keys = np.asarray(keys)
        values = np.arange(2, len(keys) + 2) if values is None else np.asarray(values)
        order = np.argsort(keys, kind="stable")
        self.keys, self.values = keys[order], values[order]

    def __call__(self, ids):
        ids = np.asarray(ids)
        if len(self.keys) == 0:
            return np.full(ids.shape, UNK)
        pos = np.minimum(np.searchsorted(self.keys, ids), len(self.keys) - 1)
        return np.where(self.keys[pos] == ids, self.values[pos], UNK)


def split_terms(strings):
    """Arrow array of TERM_SEP-joined strings -> (flat int64 term ids, terms per string).
    Empty terms (from a trailing or doubled separator, or an empty string) are dropped."""
    lists = pc.split_pattern(strings, TERM_SEP)
    terms = lists.flatten()
    keep = pc.not_equal(terms, "").to_numpy(zero_copy_only=False)
    row = np.repeat(np.arange(len(lists)), list_lengths(lists))
    lens = np.bincount(row[keep], minlength=len(lists))
    return pc.cast(terms.filter(pa.array(keep)), pa.int64()).to_numpy(), lens


def gather_ranges(starts, lens):
    """Flat positions for rows given their start offsets and lengths."""
    within = np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens)
    return np.repeat(starts, lens) + within


def pad_terms(term_idx, lens, max_terms):
    """Flat term indices + row lengths -> [n_rows, max_terms], keeping the first terms."""
    n = len(lens)
    row = np.repeat(np.arange(n), lens)
    col = np.arange(len(term_idx)) - np.repeat(np.cumsum(lens) - lens, lens)
    keep = col < max_terms
    out = np.zeros((n, max_terms), dtype=np.int32)
    out[row[keep], col[keep]] = term_idx[keep]
    return out


def decayed_recency(days, half_life_days=30.0):
    """exp(-days / tau) with tau set by a half-life. Applied at training time to hist_days
    (padding is removed by the mask), so the half-life can be tuned without rebuilding."""
    return np.exp(-np.log(2) * days / half_life_days).astype(np.float32)

## Step 1a: Load product names for every candidate

The product file is large (mostly names), so it is streamed in batches and only the
products appearing as candidates in train/val/test are kept, already truncated to
NAME_MAX_TERMS. This is catalogue lookup data, not learned, so including val/test
products here is not leakage; the term vocabulary below still comes from train only.

In [5]:
class ProductNames:
    def __init__(self, path, needed_wids, batch_size=500_000):
        needed = pa.array(np.unique(needed_wids))
        wids, terms, lens, full_lens = [], [], [], []
        for b in pq.ParquetFile(path).iter_batches(columns=["wid", "name"], batch_size=batch_size):
            b = b.filter(pc.is_in(b.column("wid"), value_set=needed.cast(b.schema.field("wid").type)))
            if b.num_rows == 0:
                continue
            name = b.column("name")
            raw = pc.cast(name.flatten(), pa.int64()).to_numpy()
            l = list_lengths(name)
            keep = (np.arange(len(raw)) - np.repeat(np.cumsum(l) - l, l)) < NAME_MAX_TERMS
            wids.append(b.column("wid").to_numpy())
            terms.append(raw[keep])
            lens.append(np.minimum(l, NAME_MAX_TERMS))
            full_lens.append(l)
        wid = np.concatenate(wids)
        lens = np.concatenate(lens)
        offsets = np.cumsum(lens) - lens
        order = np.argsort(wid, kind="stable")
        self.wid, self.lens, self.offsets = wid[order], lens[order], offsets[order]
        self.terms = np.concatenate(terms)
        full = np.concatenate(full_lens)
        print(f"product names loaded: {len(wid):,}  name length p50/p90/p95/p99 = "
              f"{np.percentile(full, [50, 90, 95, 99]).round(0).tolist()}  "
              f"(truncated at {NAME_MAX_TERMS}: {(full > NAME_MAX_TERMS).mean():.1%})")

    def raw_terms(self, wids):
        """wids -> (flat raw term ids, length per wid); unknown wids get length 0."""
        wids = np.asarray(wids)
        pos = np.minimum(np.searchsorted(self.wid, wids), len(self.wid) - 1)
        hit = self.wid[pos] == wids
        lens = np.where(hit, self.lens[pos], 0)
        return self.terms[gather_ranges(self.offsets[pos], lens)], lens

## Step 1b: Build vocabularies from the train split

Item vocabulary: items appearing at least MIN_ITEM_COUNT times in train **histories**.
Candidates are deliberately not counted: a train candidate counting itself would make
candidates look "known" more often in train than in val/test. Attribute vocabularies: brand / category / shop values of the
products seen in train. Term vocabulary: terms appearing at least TERM_MIN_COUNT times
across train history queries and train candidate names (one shared vocabulary, so a
query term and a name term with the same ID share an embedding).

In [6]:
def build_vocabs(train_table, product, names):
    hist_wids, _ = flat(train_table, "history_wid_list")
    cand_wids, _ = flat(train_table, "candidate_wid_list")
    # Items: counted in train HISTORIES only, so "known" means the same thing for train,
    # val and test candidates (a train candidate must not count itself into the vocabulary).
    h_uniq, h_counts = np.unique(hist_wids.to_numpy(), return_counts=True)
    item_vocab = h_uniq[h_counts >= MIN_ITEM_COUNT]

    prod_wid = product.column("wid").to_numpy()
    seen_in_train = np.isin(prod_wid, np.union1d(h_uniq, np.unique(cand_wids.to_numpy())))
    attr_vocabs = {col: np.unique(product.column(col).to_numpy()[seen_in_train]) for col in ATTR_COLS}

    qry = flat(train_table, "history_qry_list")[0]
    qry_terms, _ = split_terms(qry.filter(pc.not_equal(qry, NO_QUERY)))
    name_terms, _ = names.raw_terms(cand_wids.to_numpy())
    t_uniq, t_counts = np.unique(np.concatenate([qry_terms, name_terms]), return_counts=True)
    term_vocab = t_uniq[t_counts >= TERM_MIN_COUNT]

    q_set, n_set = np.unique(qry_terms), np.unique(name_terms)
    print(f"term vocab: {len(term_vocab):,} terms  |  distinct query terms {len(q_set):,}, "
          f"name terms {len(n_set):,}; share of query terms also used in names: "
          f"{np.isin(q_set, n_set).mean():.1%}")
    return item_vocab, attr_vocabs, term_vocab


class ProductAttrs:
    """wid -> attribute indices. Products without metadata, or with an attribute value
    not in the train vocabulary, get UNK."""

    def __init__(self, product, attr_vocabs):
        wid = product.column("wid").to_numpy()
        order = np.argsort(wid, kind="stable")
        self.wid = wid[order]
        self.attr_idx = {col: Lookup(attr_vocabs[col])(product.column(col).to_numpy()[order])
                         for col in ATTR_COLS}

    def __call__(self, wids):
        wids = np.asarray(wids)
        pos = np.minimum(np.searchsorted(self.wid, wids), len(self.wid) - 1)
        hit = self.wid[pos] == wids
        return {col: np.where(hit, idx[pos], UNK).astype(np.int32) for col, idx in self.attr_idx.items()}

## Step 2: Encode one split

History: each flat action gets its distance from the end of its user's history
(most recent = 0). Actions with distance < MAX_LEN are kept and scattered into
column MAX_LEN - 1 - distance, which truncates and left-pads in one vectorised step.

Past queries: each distinct query string in the split becomes one row of `qry_terms`
(row 0 = no query / padding); `hist_qry` holds the row number for each action.
Candidate names: each distinct candidate product in the split becomes one row of
`name_terms`; `cand_name` holds the row number for each candidate.

In [7]:
def encode_split(table, item_lookup, prod_attrs, term_lookup, names, max_len=MAX_LEN):
    n = table.num_rows

    # ---- history ----
    wids, lens = flat(table, "history_wid_list")
    wids = wids.to_numpy()
    row = np.repeat(np.arange(n), lens)
    pos_from_end = np.repeat(np.cumsum(lens), lens) - 1 - np.arange(len(wids))
    keep = pos_from_end < max_len
    r, c = row[keep], max_len - 1 - pos_from_end[keep]

    def scatter(values, dtype):
        out = np.zeros((n, max_len), dtype=dtype)
        out[r, c] = np.asarray(values)[keep]
        return out

    action_ids = pc.index_in(flat(table, "history_type_list")[0], value_set=pa.array(ACTIONS))
    assert action_ids.null_count == 0, "unexpected value in history_type_list"
    days = flat(table, "history_days_before")[0].to_numpy()
    has_query = flat(table, "history_has_query")[0].to_numpy(zero_copy_only=False)

    # past queries -> rows of a per-split query term table
    qry = pc.dictionary_encode(flat(table, "history_qry_list")[0])
    qry_dict = qry.dictionary
    is_noq = pc.equal(qry_dict, NO_QUERY).to_numpy(zero_copy_only=False)
    q_raw, q_lens = split_terms(qry_dict)
    q_table = pad_terms(term_lookup(q_raw).astype(np.int32), q_lens, QRY_MAX_TERMS)
    q_table[is_noq] = PAD
    qry_terms = np.vstack([np.zeros((1, QRY_MAX_TERMS), np.int32), q_table])
    q_idx = qry.indices.to_numpy() + 1
    q_idx[is_noq[q_idx - 1]] = 0

    hist = {
        "item": scatter(item_lookup(wids), np.int32),
        "action": scatter(action_ids.to_numpy() + 1, np.int8),
        "has_query": scatter(has_query, np.int8),
        "qry": scatter(q_idx, np.int32),
        "days": scatter(days, np.float32),
        "time_bucket": scatter(np.digitize(days, TIME_BINS_DAYS) + 1, np.int8),
        "len": np.minimum(lens, max_len).astype(np.int16),
        "full_len": lens.astype(np.int32),
    }
    for col, idx in prod_attrs(wids).items():
        hist[col] = scatter(idx, np.int32)

    # ---- candidates (flat, one entry per candidate) ----
    cwids, clens = flat(table, "candidate_wid_list")
    cwids = cwids.to_numpy()
    starts = np.repeat(np.cumsum(clens) - clens, clens)

    u_wids, name_row = np.unique(cwids, return_inverse=True)
    n_raw, n_lens = names.raw_terms(u_wids)
    name_terms = pad_terms(term_lookup(n_raw).astype(np.int32), n_lens, NAME_MAX_TERMS)

    cand = {
        "row": np.repeat(np.arange(n), clens).astype(np.int32),
        "pos": (np.arange(len(cwids)) - starts).astype(np.int16),
        "item": item_lookup(cwids).astype(np.int32),
        "name": name_row.astype(np.int32),
        "label": flat(table, "candidate_label_list")[0].to_numpy().astype(np.int8),
        "has_metadata": flat(table, "candidate_has_metadata")[0].to_numpy(zero_copy_only=False).astype(np.int8),
    }
    cand.update(prod_attrs(cwids))

    tables = {"qry_terms": qry_terms, "name_terms": name_terms}
    slate_id = table.column("slate_id").to_numpy()
    return slate_id, hist, cand, tables

## Step 3: Sanity checks
Also prints the unknown-item and unknown-term rates, which are worth reporting.

In [8]:
def check_split(name, table, slate_id, hist, cand, tables, max_len=MAX_LEN):
    _, lens = flat(table, "history_wid_list")
    mask = hist["action"] > 0
    assert (mask.sum(1) == np.minimum(lens, max_len)).all(), "mask length mismatch"
    assert (hist["item"][~mask] == PAD).all(), "non-zero item in padding"
    assert (mask[:, -1]).all(), "most recent action must sit in the last column"
    assert ((hist["qry"] > 0) == (hist["has_query"] == 1)).all(), "query rows disagree with has_query"

    src_labels = flat(table, "candidate_label_list")[0].to_numpy()
    assert (cand["label"] == src_labels).all(), "candidate labels changed"

    item_unk = (hist["item"][mask] == UNK).mean()
    cand_unk = (cand["item"] == UNK).mean()
    buy_unk = (cand["item"][cand["label"] == 3] == UNK).mean() if (cand["label"] == 3).any() else float("nan")
    print(f"[{name}] slates={len(slate_id):,}  candidates={len(cand['item']):,}  "
          f"truncated histories={(lens > max_len).mean():.1%}")
    print(f"[{name}] unknown items: history {item_unk:.1%} | candidates {cand_unk:.1%} | "
          f"purchased candidates {buy_unk:.1%}")
    for col in ATTR_COLS:
        print(f"[{name}] unknown {col}: candidates {(cand[col] == UNK).mean():.1%}")

    qt, nt = tables["qry_terms"], tables["name_terms"]
    no_name = (nt[cand["name"]] == PAD).all(1).mean()
    print(f"[{name}] unknown terms: queries {(qt[qt > 0] == UNK).mean():.1%} | "
          f"names {(nt[nt > 0] == UNK).mean():.1%}  |  candidates with no name: {no_name:.1%}")

## Step 4: Run and save

Train features use `train_sampled.parquet` (negative-sampled lists); val/test use the
full lists. Output per split: `<split>.npz` with `slate_id`, `hist_*`, `cand_*`,
`qry_terms` and `name_terms`, plus `vocab.json` (vocabulary sizes and settings, for
building embedding tables).

## Step 4: Run the pipeline

Train features use `train_sampled.parquet` (negative-sampled lists); val/test use the full lists, so evaluation is not made artificially easy.

### 4.1 Load the step 1 outputs

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
product_path = DATA_DIR / "product_metadata_clean.parquet"
product = pq.read_table(product_path, columns=["wid"] + ATTR_COLS)

splits = {"train": DATA_DIR / "train_sampled.parquet",
          "val": DATA_DIR / "val.parquet",
          "test": DATA_DIR / "test.parquet"}
tables = {name: read_split(path) for name, path in splits.items()}
for name, t in tables.items():
    print(f"{name}: {t.num_rows:,} slates")

train: 139,882 slates
val: 16,924 slates
test: 17,025 slates


### 4.2 Load product names for all candidates (streamed in batches)

In [10]:
all_cands = np.concatenate([flat(t, "candidate_wid_list")[0].to_numpy() for t in tables.values()])
names = ProductNames(product_path, all_cands)
del all_cands

product names loaded: 3,928,001  name length p50/p90/p95/p99 = [28.0, 39.0, 42.0, 50.0]  (truncated at 50: 0.9%)


### 4.3 Build vocabularies from train and save them

`vocab.json` records every vocabulary size (for the embedding tables) and the settings used, so the model code and the report can refer to one source.

In [11]:
item_vocab, attr_vocabs, term_vocab = build_vocabs(tables["train"], product, names)
item_lookup, term_lookup = Lookup(item_vocab), Lookup(term_vocab)
prod_attrs = ProductAttrs(product, attr_vocabs)

np.save(OUT_DIR / "item_vocab.npy", item_vocab)
np.save(OUT_DIR / "term_vocab.npy", term_vocab)
for col, v in attr_vocabs.items():
    np.save(OUT_DIR / f"{col}_vocab.npy", v)

vocab_info = {
    "n_items": int(len(item_vocab) + 2),
    **{f"n_{col}": int(len(v) + 2) for col, v in attr_vocabs.items()},
    "n_terms": int(len(term_vocab) + 2),
    "n_actions": len(ACTIONS) + 1,
    "n_time_buckets": len(TIME_BINS_DAYS) + 2,
    "max_len": MAX_LEN, "min_item_count": MIN_ITEM_COUNT,
    "term_min_count": TERM_MIN_COUNT,
    "qry_max_terms": QRY_MAX_TERMS, "name_max_terms": NAME_MAX_TERMS,
    "action_weight": ACTION_WEIGHT.tolist(), "time_bins_days": TIME_BINS_DAYS,
}
(OUT_DIR / "vocab.json").write_text(json.dumps(vocab_info, indent=2))
print(json.dumps(vocab_info, indent=2))

term vocab: 340,615 terms  |  distinct query terms 259,033, name terms 444,302; share of query terms also used in names: 51.8%
{
  "n_items": 2077334,
  "n_brand_id": 156623,
  "n_cate_id_4": 6814,
  "n_shop_id": 230116,
  "n_terms": 340617,
  "n_actions": 5,
  "n_time_buckets": 11,
  "max_len": 188,
  "min_item_count": 2,
  "term_min_count": 2,
  "qry_max_terms": 8,
  "name_max_terms": 50,
  "action_weight": [
    0.0,
    1.0,
    2.0,
    5.0,
    10.0
  ],
  "time_bins_days": [
    0.041666666666666664,
    1,
    3,
    7,
    14,
    30,
    60,
    90,
    180
  ]
}


### 4.4 Encode, check and save each split

Each split is saved as `<split>.npz` containing `slate_id`, the `hist_*` grids, the `cand_*` arrays, `qry_terms` and `name_terms`. The printed unknown-item and unknown-term rates are worth reporting in Section 1.

In [12]:
for name, table in tables.items():
    slate_id, hist, cand, term_tables = encode_split(table, item_lookup, prod_attrs, term_lookup, names)
    check_split(name, table, slate_id, hist, cand, term_tables)
    np.savez(OUT_DIR / f"{name}.npz", slate_id=slate_id,
             **{f"hist_{k}": v for k, v in hist.items()},
             **{f"cand_{k}": v for k, v in cand.items()},
             **term_tables)
    print()

[train] slates=139,882  candidates=2,988,641  truncated histories=24.9%
[train] unknown items: history 11.6% | candidates 64.4% | purchased candidates 18.3%
[train] unknown brand_id: candidates 0.5%
[train] unknown cate_id_4: candidates 0.5%
[train] unknown shop_id: candidates 0.5%
[train] unknown terms: queries 1.6% | names 0.2%  |  candidates with no name: 0.5%

[val] slates=16,924  candidates=1,511,179  truncated histories=25.7%
[val] unknown items: history 29.2% | candidates 66.9% | purchased candidates 18.6%
[val] unknown brand_id: candidates 1.0%
[val] unknown cate_id_4: candidates 0.5%
[val] unknown shop_id: candidates 0.9%
[val] unknown terms: queries 1.7% | names 0.5%  |  candidates with no name: 0.4%

[test] slates=17,025  candidates=1,500,706  truncated histories=25.0%
[test] unknown items: history 29.6% | candidates 66.8% | purchased candidates 20.0%
[test] unknown brand_id: candidates 1.0%
[test] unknown cate_id_4: candidates 0.5%
[test] unknown shop_id: candidates 1.0%
[t

## Step 5: Inspect the saved output

A quick look at what the model will receive: array names, shapes and types for the validation split.

In [13]:
data = np.load(OUT_DIR / "val.npz")
for k in data.files:
    print(f"{k:22s} {str(data[k].shape):18s} {data[k].dtype}")

slate_id               (16924,)           int32
hist_item              (16924, 188)       int32
hist_action            (16924, 188)       int8
hist_has_query         (16924, 188)       int8
hist_qry               (16924, 188)       int32
hist_days              (16924, 188)       float32
hist_time_bucket       (16924, 188)       int8
hist_len               (16924,)           int16
hist_full_len          (16924,)           int32
hist_brand_id          (16924, 188)       int32
hist_cate_id_4         (16924, 188)       int32
hist_shop_id           (16924, 188)       int32
cand_row               (1511179,)         int32
cand_pos               (1511179,)         int16
cand_item              (1511179,)         int32
cand_name              (1511179,)         int32
cand_label             (1511179,)         int8
cand_has_metadata      (1511179,)         int8
cand_brand_id          (1511179,)         int32
cand_cate_id_4         (1511179,)         int32
cand_shop_id           (1511179,)         i

## Hand-off notes for the Model 2 code

- **Join key with the scalar features (Person B):** `(slate_id, cand_pos)`; a candidate's history row is `cand_row`.
- **Mask:** a history position is real when `hist_action > 0`.
- **Computed inside the model, not stored:**
  - `weighted_action_score = ACTION_WEIGHT[hist_action]` (weights in `vocab.json`)
  - `decayed_recency(hist_days)` (half-life tuned on validation)
  - query / name vectors = mean of term embeddings over non-zero entries of `qry_terms[hist_qry]` and `name_terms[cand_name]`
- **Embedding table sizes:** read from `vocab.json`; index 0 = padding, 1 = unknown.